In [ ]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42 # for pdfs
matplotlib.rcParams['svg.fonttype'] = 'none' # for svgs
import matplotlib.pyplot as plt
import seaborn as sns

import flexiznam as flz
from cottage_analysis.pipelines import pipeline_utils
from cottage_analysis.plotting import rsof_plots
from v1_depth_map.figure_utils.treadmill import (
    load_treadmill_population_neurons_df,
    compute_treadmill_rsof_bins,
)


In [ ]:
# Set default font to Arial
import matplotlib.font_manager as fm

# optional, can be None or the path to arial.ttf:
arial_font_path = (
    "/Volumes/BlackPasspo/v1_depth_map/processed/v1_manuscript_figures/fonts/arial.ttf"
)
# "/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf"
# set matplotlib options
if arial_font_path is not None:
    arial_prop = fm.FontProperties(fname=arial_font_path)
    plt.rcParams["font.family"] = arial_prop.get_name()
    plt.rcParams.update({"mathtext.default": "regular"})  # make math mode also Arial
    fm.fontManager.addfont(arial_font_path)

In [ ]:
project = "colasa_3d-vision_revisions"
flexilims_session = flz.get_flexilims_session(project)
from v1_depth_map.paths import get_figures_roots

READ_ROOT, SAVE_ROOT = get_figures_roots(flexilims_session)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

regenerate_cached_files = False  # set to True to regenerate_cached_files everything

In [ ]:
# Load the treadmill population data (all sessions with the treadmill protocol), used
# to look up example cells by roi_uid.
(
    neurons_df_treadmill,
    simul_df_treadmill_population,
    simul_df_spheres_population,
    valid_treadmill_sessions,
    treadmill_sessions,
) = load_treadmill_population_neurons_df(flexilims_session)

In [ ]:
# Load the example session's treadmill (motorised wheel) trials, used for the two
# example-cell RS/OF tuning panels below.
EXAMPLE_SESSION = "PZAG17.3a_S20250402"
example_mouse, example_session = EXAMPLE_SESSION.split("_")

ndf, trials_df_tm, trials_df_sphere = pipeline_utils.load_treadmill_and_sphere_datasets(
    project,
    example_mouse,
    example_session,
    photodiode_protocol=5,
    filter_datasets={"anatomical_only": 3, "annotated": True},
    recording_type="two_photon",
    protocol_base_sphere="SpheresPermTubeReward",
    tread_kwargs=dict(method="plateau"),
)

max_abs_rs2motor_diff_ratio = 0.3
rs_bins, of_bins, tick_dict = compute_treadmill_rsof_bins(trials_df_tm)

In [ ]:
# Suggest example cells for EXAMPLE_SESSION: significant treadmill RS/OF (g2d) fit and
# a gaussian angle close to 45 deg (i.e. tuned to the RS/OF ratio, so depth-tuned on the
# treadmill). Sorted by how close the angle is to 45 deg, then by fit quality.
target_angle = 45  # degrees
angle_tol = 15  # keep cells within +/- this of target_angle

candidates = neurons_df_treadmill[
    (neurons_df_treadmill.session == EXAMPLE_SESSION)
    & neurons_df_treadmill.rsof_neuron_treadmill
    & (
        (neurons_df_treadmill.g2d_theta_treadmill - target_angle).abs() < angle_tol
    )
].copy()
candidates["angle_dist"] = (candidates.g2d_theta_treadmill - target_angle).abs()
candidates = candidates.sort_values(
    ["angle_dist", "rsof_test_rsq_closedloop_g2d_treadmill"],
    ascending=[True, False],
)

print(f"{len(candidates)} candidate cells in {EXAMPLE_SESSION}")
display(
    candidates[
        [
            "roi_uid",
            "g2d_theta_treadmill",
            "g2d_eccentricity_treadmill",
            "rsof_test_rsq_closedloop_g2d_treadmill",
            "g2d_preferred_RS_treadmill",
            "g2d_preferred_OF_treadmill",
            "is_depth_neuron_treadmill",
            "preferred_depth_closedloop_crossval_treadmill",
        ]
    ].head(20)
)

In [ ]:
EXAMPLE_CELL = "PZAG17.3a_S20250402_169"
EXAMPLE_CELL2 = "PZAG17.3a_S20250402_31"

In [ ]:
# 2 example cells in a single row: RS/OF tuning matrix then the stacked OF-tuning-by-RS
# panels (binned mean +/- bootstrap 95% CI and 1d gaussian fit of the trials) for cell 1,
# then the same pair for cell 2.
cm = 1 / 2.54
fig = plt.figure(figsize=(18.5 * cm, 4.5 * cm))
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xticks([])
ax.set_yticks([])

fontsize_dict = {"title": 8, "label": 7, "tick": 5, "legend": 5}

example_cells = [EXAMPLE_CELL, EXAMPLE_CELL2]
n_rs_bins = 5

# one horizontal band, all panels the same height and vertically aligned
band_bottom, band_h = 0.18, 0.75
stack_gap = 0.015  # vertical gap between two OF tuning panels
stack_h = (band_h - (n_rs_bins - 1) * stack_gap) / n_rs_bins
stack_w = 0.13
matrix_w, matrix_h = 0.185, band_h  # ~square given the figure aspect ratio
matrix_to_stack = 0.035  # gap between a matrix and its stack
cell_gap = 0.07  # gap between the two cells
block_w = matrix_w + matrix_to_stack + stack_w
x0 = 0.06  # left margin


def panel_ymax(ax):
    """Max y of the binned means + CI and of the fitted gaussian on one panel."""
    vals = []
    for container in ax.containers:  # errorbar: markers, caps, CI bars
        line, _, bars = container.lines
        vals.append(np.nanmax(line.get_ydata()))
        for bar in bars:
            segments = bar.get_segments()
            if len(segments):
                vals.append(max(np.nanmax(seg[:, 1]) for seg in segments))
    for line in ax.lines:  # the 300-point fit curve (not the markers or axhline)
        ydata = np.asarray(line.get_ydata(), dtype=float)
        if ydata.size > 100:
            vals.append(np.nanmax(ydata))
    return max(vals) if vals else np.nan


for iex, cell_uid in enumerate(example_cells):
    block_x = x0 + iex * (block_w + cell_gap)
    stack_x = block_x + matrix_w + matrix_to_stack
    example_cell = neurons_df_treadmill[neurons_df_treadmill.roi_uid == cell_uid].iloc[
        0
    ]
    roi = example_cell.roi

    # per-trial averaged rs/of/dff for this roi
    tav_df = []
    for trial, tseries in trials_df_tm.iterrows():
        ok = tseries.max_abs_rs2motor_diff_ratio_stim < max_abs_rs2motor_diff_ratio
        tav_df.append(
            dict(
                rs=tseries.RS_stim[ok].mean() * 100,
                of=np.degrees(tseries.OF_stim[ok].mean()),
                dff=tseries.dff_stim[:, roi].mean(),
            )
        )
    tav_df = pd.DataFrame(tav_df)

    # stacked OF-tuning-by-RS-bin axes, fastest RS on top
    axes_trials = [
        fig.add_axes(
            [
                stack_x,
                band_bottom + band_h - (i + 1) * stack_h - i * stack_gap,
                stack_w,
                stack_h,
            ]
        )
        for i in range(n_rs_bins)
    ]
    for b_s, b_e, ax in zip(rs_bins[2:], rs_bins[3:], axes_trials[::-1]):
        rsof_plots.plot_rsof_slice(
            ax,
            b_s,
            b_e,
            tav_df,
            of_bins[1:],
            plot_trials=False,
            niter=10,
            color=(0.1, 0.1, 0.1),
            scatter_size=20,
            linewidth=1,
            capsize=1.5,
            markersize=3,
            fontsize_dict=fontsize_dict,
        )
        ax.set_xticklabels([])
        ax.tick_params(axis="y", labelsize=fontsize_dict["tick"])
        ax.spines[["top", "right"]].set_visible(False)

    # one y scale per cell, rounded up to the next 0.5, shared with the matrix
    ytop = np.ceil(max(panel_ymax(ax) for ax in axes_trials) / 0.5) * 0.5
    for ax in axes_trials:
        ax.set_ylim(-0.1 * ytop, ytop)
        ax.set_yticks([0, ytop])
        if ax is not axes_trials[len(axes_trials) // 2]:
            ax.set_ylabel("")
    axes_trials[-1].set_xticks(
        [1, 10, 100, 1000],
        labels=["1", "10", "100", "1000"],
        fontsize=fontsize_dict["tick"],
    )
    axes_trials[-1].set_xlabel("Optic flow (°/s)", fontsize=fontsize_dict["label"])

    # RS/OF matrix heatmap, left of the stack and spanning the same band
    ax_matrix = fig.add_axes([block_x, band_bottom, matrix_w, matrix_h])
    range_kwargs = dict(
        log_range={"log_base": 2}, rs_bins=rs_bins, of_bins=of_bins, tick_dict=tick_dict
    )
    rsof_plots.plot_RS_OF_matrix(
        trials_df=trials_df_tm,
        roi=roi,
        is_closed_loop=1,
        max_abs_rs2motor_diff_ratio=max_abs_rs2motor_diff_ratio,
        ax=ax_matrix,
        fontsize_dict=fontsize_dict,
        vmin=0,
        vmax=ytop,
        cbar_width=0.01,
        title="",
        **range_kwargs,
    )

plt.savefig(SAVE_ROOT / "fig_depth_cells_examples.svg", bbox_inches="tight", dpi=300)
print(f"Saved figure to {SAVE_ROOT / 'fig_depth_cells_examples.svg'}")